# BERT-base-uncased CPT (MLM) on Indian legal data
Mirrors the Mamba-2 130M notebook: same dataset, same dev/test exclusion, same dedup, ~same tokens per step.

In [11]:
import zipfile, math, os, time, json, hashlib, pickle, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM, get_cosine_schedule_with_warmup

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available.")
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130 | CUDA: True
GPU: NVIDIA RTX 4000 Ada Generation


In [12]:
MODEL_NAME = "bert-base-uncased"
DATASET_DIR = Path("Dataset")
OUTPUT_DIR = Path("./bert-base-cpt-full")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OLD_EVAL_IDS = Path("./mamba2-130m-cpt-full/eval_ids.pkl")   # reused if present (tokenizer-independent)

NUM_ROWS = None        # None = all rows (use 500 for a pilot run)
MAX_STEPS = None       # None = full run (use 30 for a pilot run)
USE_FILENAME_KEY = True
TOKENS_PATH = OUTPUT_DIR / f"tokens_{NUM_ROWS or 'all'}.bin"

TRAIN_CSVS = [
    ("binary_multi_train.zip", "binary_multi_train/CJPE_ext_SCI_HCs_Tribunals_daily_orders_multi.csv"),
]

BLOCK_SIZE = 510        # + [CLS] + [SEP] = 512 (BERT's max positions)
MICRO_BATCH = 16        # lower if OOM
GRAD_ACCUM = 4          # 64 seqs/step * 512 = 32,768 tokens/step (same as Mamba run)
EPOCHS = 1
LEARNING_RATE = 5e-5
WARMUP_FRAC = 0.03
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
MLM_PROB = 0.15

LOG_EVERY = 10
EVAL_EVERY = 500
SAVE_EVERY = 500
SEED = 42
USE_BF16 = True

torch.manual_seed(SEED)

In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.model_max_length = int(1e9)   # silence "sequence too long" warning (we chunk ourselves)

CLS, SEP, MASK = tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.mask_token_id
VOCAB = len(tokenizer)
assert VOCAB < 65536
print("Vocab:", VOCAB, "| CLS/SEP/MASK:", CLS, SEP, MASK)

Vocab: 30522 | CLS/SEP/MASK: 101 102 103


In [14]:
def doc_key(t):
    return hashlib.md5(" ".join(t.lower().split()).encode("utf-8")).digest()

def iter_chunks(zip_name, csv_name, chunksize=2000):
    with zipfile.ZipFile(DATASET_DIR / zip_name) as z, z.open(csv_name) as f:
        for chunk in pd.read_csv(f, usecols=["filename", "text"], chunksize=chunksize):
            chunk = chunk.dropna(subset=["text"])
            chunk["text"] = chunk["text"].astype(str).str.strip()
            chunk["filename"] = chunk["filename"].astype(str)
            yield chunk[chunk["text"] != ""]

# ---- 1) dev/test documents to exclude from CPT (same set as the Mamba run) ----
EVAL_IDS_PATH = OUTPUT_DIR / "eval_ids.pkl"
if not EVAL_IDS_PATH.exists() and OLD_EVAL_IDS.exists():
    shutil.copy(OLD_EVAL_IDS, EVAL_IDS_PATH)

if EVAL_IDS_PATH.exists():
    with open(EVAL_IDS_PATH, "rb") as f:
        eval_files, eval_hashes = pickle.load(f)
else:
    eval_files, eval_hashes = set(), set()
    for zp in sorted(DATASET_DIR.glob("*.zip")):
        if not any(k in zp.name.lower() for k in ("dev", "test")):
            continue
        with zipfile.ZipFile(zp) as z:
            names = [n for n in z.namelist() if n.lower().endswith(".csv")]
        for n in names:
            for chunk in iter_chunks(zp.name, n):
                eval_files.update(chunk["filename"])
                eval_hashes.update(doc_key(t) for t in chunk["text"])
            print(f"[eval] {zp.name}:{n} | files {len(eval_files):,} | docs {len(eval_hashes):,}")
    with open(EVAL_IDS_PATH, "wb") as f:
        pickle.dump((eval_files, eval_hashes), f)
print(f"Eval documents excluded from CPT: {len(eval_hashes):,}")

# ---- 2) stream train CSVs -> dedup -> WordPiece tokenize -> uint16 file ([SEP] between docs) ----
if TOKENS_PATH.exists():
    print(f"{TOKENS_PATH.name} exists - skipping tokenization (delete it to rebuild).")
else:
    seen_files, seen_hashes = set(), set()
    n_docs = n_tokens = skipped_eval = skipped_dup = 0
    done = False
    tmp_path = TOKENS_PATH.with_suffix(".tmp")
    with open(tmp_path, "wb") as out:
        for zip_name, csv_name in TRAIN_CSVS:
            for chunk in iter_chunks(zip_name, csv_name):
                kept = []
                for t, nme in zip(chunk["text"].tolist(), chunk["filename"].tolist()):
                    k = doc_key(t)
                    if k in eval_hashes or (USE_FILENAME_KEY and nme in eval_files):
                        skipped_eval += 1
                        continue
                    if k in seen_hashes or (USE_FILENAME_KEY and nme in seen_files):
                        skipped_dup += 1
                        continue
                    seen_hashes.add(k)
                    seen_files.add(nme)
                    kept.append(t)
                if NUM_ROWS is not None:
                    kept = kept[: NUM_ROWS - n_docs]
                if kept:
                    enc = tokenizer(kept, add_special_tokens=False, truncation=False)["input_ids"]
                    arr = np.concatenate([np.asarray(ids + [SEP], dtype=np.uint16) for ids in enc])
                    arr.tofile(out)
                    n_docs += len(kept)
                    n_tokens += len(arr)
                print(f"  docs {n_docs:,} | tokens {n_tokens:,} | skipped eval {skipped_eval:,} dup {skipped_dup:,}", end="\r")
                if NUM_ROWS is not None and n_docs >= NUM_ROWS:
                    done = True
                    break
            if done:
                break
    os.replace(tmp_path, TOKENS_PATH)
    stats = dict(unique_docs=n_docs, total_tokens=n_tokens,
                 skipped_in_dev_test=skipped_eval, skipped_duplicates=skipped_dup)
    TOKENS_PATH.with_suffix(".json").write_text(json.dumps(stats, indent=2))
    print("\n", stats)

# ---- 3) memmap blocks + train/val split ----
data = np.memmap(TOKENS_PATH, dtype=np.uint16, mode="r")
n_blocks = len(data) // BLOCK_SIZE
blocks = data[: n_blocks * BLOCK_SIZE].reshape(n_blocks, BLOCK_SIZE)

def get_blocks(ids):
    ids = np.sort(np.asarray(ids))
    return torch.from_numpy(blocks[ids].astype(np.int64))

SEG = 16
n_seg = max(1, min(512, n_blocks // 20) // SEG)
starts = np.linspace(0, n_blocks - SEG, n_seg).astype(int)
val_np = np.unique(np.concatenate([np.arange(s, s + SEG) for s in starts]))
mask = np.ones(n_blocks, dtype=bool)
mask[val_np] = False
val_ids = torch.from_numpy(val_np)
train_ids = torch.from_numpy(np.flatnonzero(mask))
print(f"Blocks: {n_blocks:,} | train {len(train_ids):,} | val {len(val_ids):,} "
      f"| train tokens {len(train_ids) * (BLOCK_SIZE + 2):,}")

Eval documents excluded from CPT: 500,245
  docs 114,248 | tokens 298,067,301 | skipped eval 582,596 dup 6,101
 {'unique_docs': 114248, 'total_tokens': 298067301, 'skipped_in_dev_test': 582596, 'skipped_duplicates': 6101}
Blocks: 584,445 | train 583,933 | val 512 | train tokens 298,973,696


In [15]:
def mask_batch(x, gen=None):
    """x: (B, 510) token ids -> (input_ids, labels), both (B, 512). Standard BERT 15% / 80-10-10 masking."""
    B = x.size(0)
    x = torch.cat([torch.full((B, 1), CLS), x, torch.full((B, 1), SEP)], dim=1)
    labels = x.clone()
    special = (x == CLS) | (x == SEP)          # never mask CLS / SEP (incl. inter-document SEPs)
    prob = torch.full(x.shape, MLM_PROB)
    prob[special] = 0.0
    m = torch.bernoulli(prob, generator=gen).bool()
    labels[~m] = -100
    r = torch.rand(x.shape, generator=gen)
    inp = x.clone()
    inp[m & (r < 0.8)] = MASK                                    # 80% -> [MASK]
    rnd = m & (r >= 0.8) & (r < 0.9)                             # 10% -> random token
    inp[rnd] = torch.randint(1000, VOCAB, x.shape, generator=gen)[rnd]
    return inp, labels                                           # 10% unchanged

# sanity check
xi, yl = mask_batch(get_blocks(train_ids[:1]))
print("masked fraction:", (yl != -100).float().mean().item())
print(tokenizer.decode(xi[0][:60]))

masked fraction: 0.158203125
[CLS] to show cause within two months from the moaned of receipt of. that notice as to why the lease should not be [MASK] without any compensation under [MASK] second [MASK]viso to s. 20 ( 1 ) of the abolition act. the appellant showed [MASK] which [MASK] did not satisfy the said authorities [MASK]


In [16]:
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
model.to(device)
model.train()
print("Parameters:", sum(p.numel() for p in model.parameters()))

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 229.35it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Parameters: 109514298


In [17]:
decay, no_decay = [], []
for p in model.parameters():
    if p.requires_grad:
        (decay if p.ndim >= 2 else no_decay).append(p)

optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": WEIGHT_DECAY},
     {"params": no_decay, "weight_decay": 0.0}],
    lr=LEARNING_RATE, betas=(0.9, 0.98),
)

seqs_per_step = MICRO_BATCH * GRAD_ACCUM
steps_per_epoch = len(train_ids) // seqs_per_step
total_steps = steps_per_epoch * EPOCHS
if MAX_STEPS is not None:
    total_steps = min(total_steps, MAX_STEPS)

scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=max(1, int(WARMUP_FRAC * total_steps)),
    num_training_steps=total_steps,
)
print(f"Steps/epoch: {steps_per_epoch:,} | total steps: {total_steps:,} | "
      f"tokens/step: {seqs_per_step * (BLOCK_SIZE + 2):,}")

Steps/epoch: 9,123 | total steps: 9,123 | tokens/step: 32,768


In [18]:
autocast_ctx = lambda: torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=USE_BF16)

@torch.no_grad()
def evaluate():
    model.eval()
    gen = torch.Generator().manual_seed(0)      # fixed masks -> comparable across evaluations
    losses = []
    for i in range(0, len(val_ids), MICRO_BATCH * 2):
        inp, lab = mask_batch(get_blocks(val_ids[i:i + MICRO_BATCH * 2]), gen)
        with autocast_ctx():
            out = model(input_ids=inp.to(device), labels=lab.to(device))
        losses.append(out.loss.item())
    model.train()
    loss = float(np.mean(losses))
    return loss, math.exp(loss)

CKPT_PATH = OUTPUT_DIR / "ckpt.pt"

def save_checkpoint(step):
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "step": step},
               CKPT_PATH.with_suffix(".tmp"))
    os.replace(CKPT_PATH.with_suffix(".tmp"), CKPT_PATH)

start_step = 0
if CKPT_PATH.exists():
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    start_step = ckpt["step"]
    print(f"Resumed from step {start_step}")
    del ckpt

if start_step == 0:
    v_loss, v_ppl = evaluate()
    print(f"Baseline (before CPT) | val MLM loss {v_loss:.4f} | pseudo-ppl {v_ppl:.2f}")

Baseline (before CPT) | val MLM loss 2.6365 | pseudo-ppl 13.96


In [19]:
print("\nStarting CPT...\n")
log_path = OUTPUT_DIR / "train_log.csv"
if start_step == 0:
    log_path.write_text("step,train_loss,lr,tokens_per_sec,val_loss,val_ppl\n")

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
run_loss, run_steps, run_tokens = 0.0, 0, 0
t_log = time.time()
cached_epoch, order = -1, None
tok_per_seq = BLOCK_SIZE + 2

for step in range(start_step, total_steps):
    epoch, j = divmod(step, steps_per_epoch)
    if epoch != cached_epoch:
        order = torch.randperm(len(train_ids), generator=torch.Generator().manual_seed(SEED + epoch))
        cached_epoch = epoch
    idx = order[j * seqs_per_step:(j + 1) * seqs_per_step]

    optimizer.zero_grad(set_to_none=True)
    step_loss = 0.0
    for m in range(GRAD_ACCUM):
        x = get_blocks(train_ids[idx[m * MICRO_BATCH:(m + 1) * MICRO_BATCH]])
        inp, lab = mask_batch(x)
        with autocast_ctx():
            out = model(input_ids=inp.to(device), labels=lab.to(device))
        loss = out.loss / GRAD_ACCUM
        loss.backward()
        step_loss += loss.item()

    if not math.isfinite(step_loss):
        raise RuntimeError(f"Non-finite loss at step {step}. Try USE_BF16=False or a lower LEARNING_RATE.")

    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    optimizer.step()
    scheduler.step()

    run_loss += step_loss
    run_steps += 1
    run_tokens += seqs_per_step * tok_per_seq
    done = step + 1

    if done % LOG_EVERY == 0 or done == total_steps:
        now = time.time()
        tps = run_tokens / (now - t_log)
        eta_min = (total_steps - done) * seqs_per_step * tok_per_seq / tps / 60
        lr = scheduler.get_last_lr()[0]
        avg = run_loss / run_steps
        print(f"step {done}/{total_steps} | loss {avg:.4f} | lr {lr:.2e} | "
              f"{tps:,.0f} tok/s | ETA {eta_min:,.1f} min | "
              f"peak mem {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
        with open(log_path, "a") as f:
            f.write(f"{done},{avg:.5f},{lr:.3e},{tps:.0f},,\n")
        run_loss, run_steps, run_tokens, t_log = 0.0, 0, 0, now

    if done % EVAL_EVERY == 0 or done == total_steps:
        v_loss, v_ppl = evaluate()
        print(f"  >> val MLM loss {v_loss:.4f} | pseudo-ppl {v_ppl:.2f}")
        with open(log_path, "a") as f:
            f.write(f"{done},,,,{v_loss:.5f},{v_ppl:.3f}\n")

    if done % SAVE_EVERY == 0:
        save_checkpoint(done)

print(f"\nTraining finished in {(time.time() - t0) / 60:.1f} min")


Starting CPT...

step 10/9123 | loss 2.7464 | lr 1.83e-06 | 39,042 tok/s | ETA 127.5 min | peak mem 7.74 GB
step 20/9123 | loss 2.7636 | lr 3.66e-06 | 39,573 tok/s | ETA 125.6 min | peak mem 7.74 GB
step 30/9123 | loss 2.6612 | lr 5.49e-06 | 39,610 tok/s | ETA 125.4 min | peak mem 7.74 GB
step 40/9123 | loss 2.5638 | lr 7.33e-06 | 39,463 tok/s | ETA 125.7 min | peak mem 7.74 GB
step 50/9123 | loss 2.4769 | lr 9.16e-06 | 39,381 tok/s | ETA 125.8 min | peak mem 7.74 GB
step 60/9123 | loss 2.4423 | lr 1.10e-05 | 39,307 tok/s | ETA 125.9 min | peak mem 7.74 GB
step 70/9123 | loss 2.3818 | lr 1.28e-05 | 39,085 tok/s | ETA 126.5 min | peak mem 7.74 GB
step 80/9123 | loss 2.3626 | lr 1.47e-05 | 37,289 tok/s | ETA 132.4 min | peak mem 7.74 GB
step 90/9123 | loss 2.3411 | lr 1.65e-05 | 37,103 tok/s | ETA 133.0 min | peak mem 7.74 GB
step 100/9123 | loss 2.2959 | lr 1.83e-05 | 37,043 tok/s | ETA 133.0 min | peak mem 7.74 GB
step 110/9123 | loss 2.2968 | lr 2.01e-05 | 36,878 tok/s | ETA 133.5 mi

In [20]:
final_dir = OUTPUT_DIR / "final"
final_dir.mkdir(exist_ok=True)
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)
print("Saved to", final_dir.resolve())
print("Peak GPU memory allocated:", round(torch.cuda.max_memory_allocated() / 1024**3, 2), "GB")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.59it/s]

Saved to C:\Users\HP\Desktop\2022IMT-061 Priya Keshri MTP\bert-base-cpt-full\final
Peak GPU memory allocated: 7.74 GB
